# x64 Secondary Holdout 70k 3p5k Build

This notebook regenerates the secondary `70k` normal and `3.5k` defect holdout used for larger-scale evaluation.

Unlike the main benchmark notebook, this branch preserves the original `50k 5%` train and validation rows and only replaces the test split with a larger disjoint holdout.


## Notebook Flow

Run the notebook from top to bottom.

1. load the holdout config for this branch
2. confirm the raw pickle and base `50k 5%` metadata exist
3. run the shared secondary-holdout builder
4. validate the generated holdout metadata and arrays
5. inspect the resulting split summary and defect breakdown


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATASET_DIR = REPO_ROOT / "data" / "dataset" / "x64" / "holdout70k_3p5k"
CONFIG_PATH = DATASET_DIR / "data_config.toml"

def repo_path(path_like: str | Path) -> Path:
    path = Path(path_like)
    return path if path.is_absolute() else (REPO_ROOT / path)

REPO_ROOT, CONFIG_PATH


In [ ]:
from wafer_defect.config import load_toml

config = load_toml(CONFIG_PATH)
holdout_cfg = config["holdout"]

raw_pickle = repo_path(holdout_cfg["raw_pickle"])
base_metadata_path = repo_path(holdout_cfg["base_metadata_csv"])
output_metadata_path = repo_path(holdout_cfg["output_metadata_csv"])
output_arrays_dir = output_metadata_path.parent / f"arrays_{output_metadata_path.stem.removeprefix('metadata_')}"

config_summary = pd.DataFrame(
    [
        {"field": "raw_pickle", "value": holdout_cfg["raw_pickle"]},
        {"field": "base_metadata_csv", "value": holdout_cfg["base_metadata_csv"]},
        {"field": "output_metadata_csv", "value": holdout_cfg["output_metadata_csv"]},
        {"field": "image_size", "value": holdout_cfg["image_size"]},
        {"field": "test_normal_count", "value": holdout_cfg["test_normal_count"]},
        {"field": "test_defect_count", "value": holdout_cfg["test_defect_count"]},
        {"field": "random_seed", "value": holdout_cfg["random_seed"]},
        {"field": "expected_arrays_dir", "value": output_arrays_dir.relative_to(REPO_ROOT).as_posix()},
    ]
)
display(config_summary)


## Input Checks

The holdout builder depends on two existing inputs:

- the raw WM-811K pickle
- the previously generated base metadata file `metadata_50k_5pct.csv`


In [ ]:
if not raw_pickle.exists():
    raise FileNotFoundError(f"Raw dataset file not found: {raw_pickle}")
if not base_metadata_path.exists():
    raise FileNotFoundError(f"Base metadata file not found: {base_metadata_path}")

base_metadata = pd.read_csv(base_metadata_path)
display(base_metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index().sort_values(["split", "is_anomaly"]))


## Generate the Secondary Holdout

This cell runs the dedicated holdout builder script and writes the new metadata and arrays into `data/processed/x64/wm811k/`.


In [ ]:
command = [
    sys.executable,
    "scripts/build_secondary_holdout_metadata.py",
    "--raw-pickle",
    str(Path(holdout_cfg["raw_pickle"])),
    "--base-metadata",
    str(Path(holdout_cfg["base_metadata_csv"])),
    "--output-metadata",
    str(Path(holdout_cfg["output_metadata_csv"])),
    "--test-normal-count",
    str(holdout_cfg["test_normal_count"]),
    "--test-defect-count",
    str(holdout_cfg["test_defect_count"]),
    "--image-size",
    str(holdout_cfg["image_size"]),
    "--seed",
    str(holdout_cfg["random_seed"]),
]
pythonpath_entries = [str(REPO_ROOT), str(REPO_ROOT / "src")]
existing_pythonpath = os.environ.get("PYTHONPATH")
if existing_pythonpath:
    pythonpath_entries.append(existing_pythonpath)
run_env = os.environ.copy()
run_env["PYTHONPATH"] = os.pathsep.join(pythonpath_entries)

result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
    env=run_env,
)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)


## Validate the Written Holdout Outputs


In [ ]:
if not output_metadata_path.exists():
    raise FileNotFoundError(f"Expected holdout metadata file was not created: {output_metadata_path}")
if not output_arrays_dir.exists():
    raise FileNotFoundError(f"Expected holdout arrays directory was not created: {output_arrays_dir}")

holdout_metadata = pd.read_csv(output_metadata_path)
display(holdout_metadata.head())
display(holdout_metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index().sort_values(["split", "is_anomaly"]))

array_files = sorted(output_arrays_dir.glob("*.npy"))
array_file_names = {path.name for path in array_files}
test_rows = holdout_metadata[holdout_metadata["split"] == "test"].copy()
metadata_file_names = set(test_rows["array_path"].map(lambda value: Path(value).name))
missing_arrays = sorted(metadata_file_names - array_file_names)
extra_arrays = sorted(array_file_names - metadata_file_names)

validation_df = pd.DataFrame([
    {
        "total_metadata_rows": len(holdout_metadata),
        "test_rows": len(test_rows),
        "array_files": len(array_files),
        "missing_arrays": len(missing_arrays),
        "extra_arrays": len(extra_arrays),
    }
])
display(validation_df)

assert len(missing_arrays) == 0, f"Metadata references missing arrays, e.g. {missing_arrays[:5]}"
assert len(extra_arrays) == 0, f"Arrays directory contains unexpected extra files, e.g. {extra_arrays[:5]}"


In [ ]:
holdout_test = holdout_metadata[holdout_metadata["split"] == "test"].copy()
display(
    holdout_test[holdout_test["is_anomaly"] == 1]["defect_type"]
    .value_counts()
    .rename_axis("defect_type")
    .to_frame("count")
)

sample_rows = holdout_test.sample(n=min(6, len(holdout_test)), random_state=int(holdout_cfg["random_seed"]))
sample_shapes = []
for _, row in sample_rows.iterrows():
    wafer = np.load(repo_path(row["array_path"]))
    sample_shapes.append({
        "array_path": row["array_path"],
        "shape": tuple(wafer.shape),
        "min": float(wafer.min()),
        "max": float(wafer.max()),
    })
display(pd.DataFrame(sample_shapes))


## Why This Notebook Exists

The holdout protocol is a separate dataset branch, not just a different evaluation command. Giving it its own notebook makes the train/val-preserving split logic reproducible for graders and keeps it separate from the main benchmark generation notebook.
